In [7]:
# MEXC — Authenticated WS Depth Test
# Tests whether the "Blocked" error is IP-based or auth-related.
# Paste into a single Jupyter cell and run.

import os
import asyncio, json, time, hmac, hashlib, aiohttp
from collections import OrderedDict
from urllib.parse import urlencode

TRADING_PAIR = "TONUSDT"
SYMBOL = "TONUSDT"
REST_BASE = "https://api.mexc.com"
WS_URL = "wss://wbs-api.mexc.com/ws"

# Fill in your MEXC API credentials
API_KEY = os.environ["MEXC_API_KEY"]
API_SECRET = os.environ["MEXC_API_SECRET"]

if not API_KEY or not API_SECRET:
    print("ERROR: Set API_KEY and API_SECRET above")
    raise SystemExit
 
print("=" * 60)
print("Testing MEXC WS with CORRECT channel format")
print("(matching your working candle ingest)")
print("=" * 60)
print()
 
# These are the EXACT formats from candle_ws_mexc.py that WORK:
CORRECT_CHANNELS = [
    f"spot@public.aggre.deals.v3.api.pb@100ms@{SYMBOL}",       # trades
    f"spot@public.aggre.bookTicker.v3.api.pb@100ms@{SYMBOL}",  # book ticker
    f"spot@public.kline.v3.api.pb@{SYMBOL}@Min1",              # 1m kline
]
 
# This is what Hummingbot uses for depth (NOT in candle ingest):
HBOT_DEPTH_CHANNEL = f"spot@public.aggre.depth.v3.api.pb@100ms@{SYMBOL}"
 
# These are the WRONG formats our earlier tests used:
WRONG_CHANNELS = [
    f"spot@public.increase.depth.v3.api@{SYMBOL}@100ms",  # wrong format
    f"spot@public.deals.v3.api@{SYMBOL}",                  # missing aggre, pb, 100ms
    f"spot@public.bookTicker.v3.api@{SYMBOL}",             # missing aggre, pb, 100ms
]
 
print("CORRECT channels (from candle ingest):")
for c in CORRECT_CHANNELS:
    print(f"  {c}")
print(f"\nHummingbot depth channel:")
print(f"  {HBOT_DEPTH_CHANNEL}")
print(f"\nWRONG channels (what our earlier tests used):")
for c in WRONG_CHANNELS:
    print(f"  {c}")
 
# ---------- Test 1: Correct format channels ----------
print()
print("=" * 60)
print("TEST 1: Correct channels (candle ingest format)")
print("=" * 60)
 
async def test_correct_channels():
    try:
        async with aiohttp.ClientSession() as session:
            async with session.ws_connect(WS_URL, timeout=15) as ws:
                sub = {"method": "SUBSCRIPTION", "params": CORRECT_CHANNELS, "id": 1}
                await ws.send_json(sub)
                print(f"Sent subscribe for {len(CORRECT_CHANNELS)} channels")
 
                binary_count = 0
                text_count = 0
                start = time.time()
 
                while time.time() - start < 15:
                    try:
                        msg = await asyncio.wait_for(ws.receive(), timeout=5)
                    except asyncio.TimeoutError:
                        print("  (timeout)")
                        break
 
                    if msg.type == aiohttp.WSMsgType.TEXT:
                        data = json.loads(msg.data)
                        text_count += 1
                        blocked = "Blocked" in str(data) or "Not Subscribed" in str(data)
                        if blocked:
                            print(f"  BLOCKED: {json.dumps(data)[:300]}")
                        else:
                            print(f"  OK (text #{text_count}): {json.dumps(data)[:200]}")
 
                    elif msg.type == aiohttp.WSMsgType.BINARY:
                        binary_count += 1
                        if binary_count <= 3:
                            print(f"  OK (binary #{binary_count}): {len(msg.data)} bytes ← DATA FLOWING!")
                        elif binary_count == 4:
                            print(f"  ... (receiving more binary frames, stopping log)")
 
                    elif msg.type in (aiohttp.WSMsgType.CLOSED, aiohttp.WSMsgType.ERROR):
                        print(f"  Closed/Error")
                        break
 
                print(f"\nResult: {binary_count} binary frames, {text_count} text frames")
                return binary_count > 0
 
    except Exception as e:
        print(f"ERROR: {type(e).__name__}: {e}")
        return False
 
correct_works = await test_correct_channels()
 
# ---------- Test 2: Hummingbot depth channel ----------
print()
print("=" * 60)
print("TEST 2: Hummingbot depth channel (aggre.depth)")
print("=" * 60)
 
async def test_hbot_depth():
    try:
        async with aiohttp.ClientSession() as session:
            async with session.ws_connect(WS_URL, timeout=15) as ws:
                sub = {"method": "SUBSCRIPTION", "params": [HBOT_DEPTH_CHANNEL], "id": 2}
                await ws.send_json(sub)
                print(f"Sent subscribe: {HBOT_DEPTH_CHANNEL}")
 
                binary_count = 0
                start = time.time()
 
                while time.time() - start < 12:
                    try:
                        msg = await asyncio.wait_for(ws.receive(), timeout=5)
                    except asyncio.TimeoutError:
                        print("  (timeout)")
                        break
 
                    if msg.type == aiohttp.WSMsgType.TEXT:
                        data = json.loads(msg.data)
                        blocked = "Blocked" in str(data) or "Not Subscribed" in str(data)
                        if blocked:
                            print(f"  BLOCKED: {json.dumps(data)[:300]}")
                        else:
                            print(f"  OK (text): {json.dumps(data)[:200]}")
 
                    elif msg.type == aiohttp.WSMsgType.BINARY:
                        binary_count += 1
                        if binary_count <= 3:
                            print(f"  OK (binary #{binary_count}): {len(msg.data)} bytes")
 
                    elif msg.type in (aiohttp.WSMsgType.CLOSED, aiohttp.WSMsgType.ERROR):
                        break
 
                print(f"\nResult: {binary_count} binary frames")
                return binary_count > 0
 
    except Exception as e:
        print(f"ERROR: {type(e).__name__}: {e}")
        return False
 
depth_works = await test_hbot_depth()
 
# ---------- Test 3: Wrong format (confirm it's the format, not IP) ----------
print()
print("=" * 60)
print("TEST 3: Wrong format (confirm BLOCKED is format-related)")
print("=" * 60)
 
async def test_wrong_channels():
    try:
        async with aiohttp.ClientSession() as session:
            async with session.ws_connect(WS_URL, timeout=10) as ws:
                sub = {"method": "SUBSCRIPTION", "params": WRONG_CHANNELS, "id": 3}
                await ws.send_json(sub)
                print(f"Sent subscribe with WRONG format")
 
                start = time.time()
                while time.time() - start < 8:
                    try:
                        msg = await asyncio.wait_for(ws.receive(), timeout=4)
                    except asyncio.TimeoutError:
                        break
 
                    if msg.type == aiohttp.WSMsgType.TEXT:
                        data = json.loads(msg.data)
                        blocked = "Blocked" in str(data) or "Not Subscribed" in str(data)
                        print(f"  {'BLOCKED' if blocked else 'OK'}: {json.dumps(data)[:300]}")
 
                    elif msg.type == aiohttp.WSMsgType.BINARY:
                        print(f"  Binary: {len(msg.data)} bytes")
 
    except Exception as e:
        print(f"ERROR: {e}")
 
await test_wrong_channels()
 
# ---------- Summary ----------
print()
print("=" * 60)
print("CONCLUSION")
print("=" * 60)
print(f"Correct candle-ingest format works? {correct_works}")
print(f"Hummingbot depth channel works?     {depth_works}")
print()
if correct_works and not depth_works:
    print(">>> Candle ingest format works but Hummingbot depth doesn't.")
    print(">>> The aggre.depth channel may be blocked or has different format.")
    print(">>> Consider if Hummingbot should use bookTicker instead of full depth.")
elif correct_works and depth_works:
    print(">>> BOTH work! The 'Blocked' errors were from WRONG channel names.")
    print(">>> NOT an IP/region block — just bad channel format in our tests.")
    print(">>> No VPN change needed!")
elif not correct_works:
    print(">>> Even correct format is blocked — this IS an IP/connection limit issue.")
    print(">>> May need to stop candle ingest before testing, or change VPN region.")
print()
print(">>> PASTE OUTPUT BACK TO CLAUDE <<<")

Testing MEXC WS with CORRECT channel format
(matching your working candle ingest)

CORRECT channels (from candle ingest):
  spot@public.aggre.deals.v3.api.pb@100ms@TONUSDT
  spot@public.aggre.bookTicker.v3.api.pb@100ms@TONUSDT
  spot@public.kline.v3.api.pb@TONUSDT@Min1

Hummingbot depth channel:
  spot@public.aggre.depth.v3.api.pb@100ms@TONUSDT

WRONG channels (what our earlier tests used):
  spot@public.increase.depth.v3.api@TONUSDT@100ms
  spot@public.deals.v3.api@TONUSDT
  spot@public.bookTicker.v3.api@TONUSDT

TEST 1: Correct channels (candle ingest format)
Sent subscribe for 3 channels
  OK (text #1): {"id": 1, "code": 0, "msg": "spot@public.aggre.bookTicker.v3.api.pb@100ms@TONUSDT,spot@public.aggre.deals.v3.api.pb@100ms@TONUSDT,spot@public.kline.v3.api.pb@TONUSDT@Min1"}
  OK (binary #1): 105 bytes ← DATA FLOWING!
  OK (binary #2): 105 bytes ← DATA FLOWING!
  OK (binary #3): 105 bytes ← DATA FLOWING!
  ... (receiving more binary frames, stopping log)

Result: 151 binary frames, 1 